# Normalization and transformation inventory: H1-H4
Every column in the current matched merged CSV is inventoried, including sound and SVF, plus six bilateral means used by H4. Physiological transformations preserve the existing model contract. Native scale, nonlinear transformation and model-specific standardization are separate decisions. Environmental literature is reviewed by parameter group; source methods and current modelling choices are distinguished. Zero cited papers means this notebook has no mapped transformation reference for that field. Distribution diagnostics do not establish residual-model validity. Existing paper mappings are carried forward, not a new full-text systematic review.
The final environmental table separates nonlinear transformation from centering/scaling and checks sound inputs against the PCA script. It documents current specifications rather than claiming optimal transformations for every model.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import importlib.util
import re
import os
import shutil
import tempfile


In [2]:
# 1. Config
ROOT = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final")
DATA = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final/output/merged_all_11participants_8min.csv")  # same input as H1_to_H3.R / H4_multivariate.R
CORE_PHASES = ["BikeU", "BikeG", "WalkU", "WalkG"]  # x <= 0 counts use the analysed core phases only
PAPERS = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Github/Experiment papers")
OUT_DIR = ROOT / "output/statistical_analysis/Normalization & Parameter Screening/Normalization"
OUT_COUNTS = OUT_DIR / "Normalization_H1toH4_counts.csv"
OUT_INDEX = OUT_DIR / "Normalization_H1toH4_paper_index.csv"
OUT_PHYSIO = OUT_DIR / "Normalization_H1toH4_physio.csv"
OUT_ROLES = OUT_DIR / "Normalization_H1toH4_paper_roles.csv"

# Wording of each transformation as modelled. HRV: ln(x), rows <= 0 excluded (Laborde, Mosley & Thayer 2017);
# never ln(1 + x). Must stay identical to H4_SPEC$outcome_transform in scripts/H4_multivariate.R.
STEP5_TEXT = {
    "No transformation": "No transformation",
    "Natural log": "Natural log",
}
# Physiological outcomes fitted in H4 Step 5 (H4_SPEC$outcomes in scripts/H4_multivariate.R).
# empatica__eda_scl_usiemens is not a Step 5 outcome.
NOT_IN_STEP5 = {"empatica__eda_scl_usiemens"}

# Current physiological rule and mapped references. Reference counts are mappings, not verified endorsements of that rule.

PAPER_GROUP_1 = ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "Zhang et al 2020", "Figner_Murphy_2010"]
PAPER_GROUP_2 = ["Laborde Mosley Thayer 2017", "An Overview of Heart Rate Variability Metrics and Norms", "HRV_task_force", "Reliability of Ultra-Short-Term", "Threshold-Based Artefact Correction"]
PAPER_GROUP_3 = ["HRV_task_force", "An Overview of Heart Rate Variability Metrics and Norms", "Laborde Mosley Thayer 2017", "Reliability of Ultra-Short-Term", "Threshold-Based Artefact Correction"]
PAPER_GROUP_4 = ["Imburgio et al 2024", "Beggiato et al 2018", "Mahanama et al 2021", "Wang et al 2025", "Wang et al 2024"]
PAPER_GROUP_5 = ["Wang et al 2025", "Wang et al 2024", "Evans et al 2012", "Tomasi et al 2016", "Rupi and Krizek 2019", "Trefzger et al 2018", "Vansteenkiste et al 2014"]
PAPER_GROUP_6 = ["Wang et al 2025", "Wang et al 2024", "Evans et al 2012", "Tomasi et al 2016", "Fleming et al 2024"]

USED = {
    "Heart rate": [("No transformation", ["Zhang et al 2020", "Beggiato et al 2018", "Outdoor temperature heart rate", "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Tervonen"])],
    "Skin temperature": [("No transformation", ["Zhang et al 2020", "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Kyriakou", "novel methodology for human thermal comfort"])],
    "Skin conductance level": [("No transformation", PAPER_GROUP_1)],
    "Skin conductance, tonic": [("No transformation", PAPER_GROUP_1)],
    "Skin conductance, phasic": [("No transformation", ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "EDA_transform_BIOPAC_EDA_SCR", "Figner_Murphy_2010"])],
    "SDNN": [("Natural log", PAPER_GROUP_2)],
    "RMSSD": [("Natural log", PAPER_GROUP_2)],
    "LF power": [("Natural log", PAPER_GROUP_2)],
    "HF power": [("Natural log", PAPER_GROUP_2)],
    "VLF power": [("Natural log", PAPER_GROUP_2)],
    "Total power": [("Natural log", PAPER_GROUP_2)],
    "LF/HF ratio": [("Natural log", PAPER_GROUP_2)],
    "LF normalized": [("No transformation", PAPER_GROUP_3)],
    "HF normalized": [("No transformation", PAPER_GROUP_3)],
    "Pupil diameter": [("No transformation", PAPER_GROUP_4)],
    "Pupil change rate": [("No transformation", PAPER_GROUP_4)],
    "Eyelid aperture": [("No transformation", ["Mahanama et al 2021", "Beggiato et al 2018", "Wang et al 2025", "Wang et al 2024", "Evans et al 2012"])],
    "Blink duration": [("No transformation", PAPER_GROUP_5)],
    "Fixation duration": [("No transformation", PAPER_GROUP_5)],
    "Saccade duration": [("No transformation", PAPER_GROUP_5)],
    "Fixation rate": [("No transformation", PAPER_GROUP_5)],
    "Saccade amplitude": [("No transformation", PAPER_GROUP_5)],
    "Saccade peak velocity": [("No transformation", PAPER_GROUP_5)],
    "Gaze velocity": [("No transformation", PAPER_GROUP_5)],
    "Gaze dispersion": [("No transformation", PAPER_GROUP_5)],
    "Distance from center": [("No transformation", PAPER_GROUP_6)],
    "Gaze centrality": [("No transformation", PAPER_GROUP_6)],
}

COL = {
    "Heart rate": "empatica_heart_rate",
    "Skin temperature": "empatica_skin_temperature",
    "Skin conductance level": "empatica__eda_scl_usiemens",
    "Skin conductance, tonic": "empatica_eda_tonic",
    "Skin conductance, phasic": "empatica_eda_phasic",
    "SDNN": "empatica_hrv_td_sdnn",
    "RMSSD": "empatica_hrv_td_rmssd",
    "LF power": "empatica_hrv_fd_lf",
    "HF power": "empatica_hrv_fd_hf",
    "VLF power": "empatica_hrv_fd_vlf",
    "Total power": "empatica_hrv_fd_total_power",
    "LF/HF ratio": "empatica_hrv_fd_lf_hf_ratio",
    "LF normalized": "empatica_hrv_fd_lfnu",
    "HF normalized": "empatica_hrv_fd_hfnu",
    "Pupil diameter": "eyetracker_pupil_diameter_avg",
    "Pupil change rate": "eyetracker_pupil_change_rate",
    "Eyelid aperture": "eyetracker_eyelid_aperture_avg",
    "Blink duration": "eyetracker_blink_duration_s",
    "Fixation duration": "eyetracker_fixation_duration_s",
    "Saccade duration": "eyetracker_saccade_duration_s",
    "Fixation rate": "eyetracker_fixation_rate",
    "Saccade amplitude": "eyetracker_saccade_amplitude",
    "Saccade peak velocity": "eyetracker_saccade_peak_velocity",
    "Gaze velocity": "eyetracker_gaze_velocity",
    "Gaze dispersion": "eyetracker_gaze_dispersion",
    "Distance from center": "eyetracker_distance_from_center",
    "Gaze centrality": "eyetracker_gaze_centrality",
}

UNIT = {
    "empatica_heart_rate": "bpm", "empatica_skin_temperature": "°C", "empatica__eda_scl_usiemens": "microsiemens",
    "empatica_eda_tonic": "microsiemens", "empatica_eda_phasic": "microsiemens", "empatica_hrv_td_sdnn": "ms",
    "empatica_hrv_td_rmssd": "ms", "empatica_hrv_fd_lf": "ms2", "empatica_hrv_fd_hf": "ms2", "empatica_hrv_fd_vlf": "ms2",
    "empatica_hrv_fd_total_power": "ms2", "empatica_hrv_fd_lf_hf_ratio": "ratio", "empatica_hrv_fd_lfnu": "%",
    "empatica_hrv_fd_hfnu": "%", "eyetracker_pupil_diameter_avg": "mm", "eyetracker_pupil_change_rate": "mm/s",
    "eyetracker_eyelid_aperture_avg": "mm", "eyetracker_blink_duration_s": "s", "eyetracker_fixation_duration_s": "s",
    "eyetracker_saccade_duration_s": "s", "eyetracker_fixation_rate": "per second", "eyetracker_saccade_amplitude": "degrees",
    "eyetracker_saccade_peak_velocity": "degrees/s", "eyetracker_gaze_velocity": "degrees/s",
    "eyetracker_gaze_dispersion": "degrees", "eyetracker_distance_from_center": "pixels", "eyetracker_gaze_centrality": "proportion",
}
assert set(USED) == set(COL), set(USED) ^ set(COL)
assert set(COL.values()) == set(UNIT), set(COL.values()) ^ set(UNIT)
assert {t for opts in USED.values() for t, _ in opts} <= set(STEP5_TEXT)

# Literature transformation voting: related parameters share one vote group and one Step 5 rule.
# Paper keys must exist in paper_paths; environmental IDs must exist in ENVIRONMENT_LITERATURE.
PHYSIO_LITERATURE_VOTES = [
    {"vote_group": "Heart rate", "parameters": ["Heart rate"],
     "candidates": [("No transformation", ["Zhang et al 2020", "Beggiato et al 2018", "Outdoor temperature heart rate",
        "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Tervonen"])]},
    {"vote_group": "Skin temperature", "parameters": ["Skin temperature"],
     "candidates": [("No transformation", ["Zhang et al 2020", "Personal_thermal_comfort_wearable_sensors_2018",
        "Thermal_comfort_physiological_sensing_DTU_2025", "Kyriakou", "novel methodology for human thermal comfort"])]},
    {"vote_group": "EDA (SCL, tonic, phasic)", "parameters": ["Skin conductance level", "Skin conductance, tonic", "Skin conductance, phasic"],
     "candidates": [
        ("Log or square-root (component-specific)", ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "Figner_Murphy_2010", "EDA_transform_BIOPAC_EDA_SCR"]),
        ("Native scale / no nonlinear transform", ["Zhang et al 2020"]),
     ]},
    {"vote_group": "HRV absolute power, SDNN, RMSSD, LF/HF ratio",
     "parameters": ["SDNN", "RMSSD", "LF power", "HF power", "VLF power", "Total power", "LF/HF ratio"],
     "candidates": [("Natural log", PAPER_GROUP_2)]},
    {"vote_group": "HRV normalized (LFnu, HFnu)", "parameters": ["LF normalized", "HF normalized"],
     "candidates": [("No transformation", PAPER_GROUP_3)]},
    {"vote_group": "Pupil metrics", "parameters": ["Pupil diameter", "Pupil change rate"],
     "candidates": [("No transformation", PAPER_GROUP_4)]},
    {"vote_group": "Eyelid aperture", "parameters": ["Eyelid aperture"],
     "candidates": [("No transformation", ["Mahanama et al 2021", "Beggiato et al 2018", "Wang et al 2025", "Wang et al 2024", "Evans et al 2012"])]},
    {"vote_group": "Gaze event timings and kinematics",
     "parameters": ["Blink duration", "Fixation duration", "Saccade duration", "Fixation rate", "Saccade amplitude",
                    "Saccade peak velocity", "Gaze velocity", "Gaze dispersion"],
     "candidates": [("No transformation", PAPER_GROUP_5)]},
    {"vote_group": "Gaze centrality and distance", "parameters": ["Distance from center", "Gaze centrality"],
     "candidates": [("No transformation", PAPER_GROUP_6)]},
]

ENV_H4_NONLINEAR = {
    "Thermal": "None; native scale",
    "Pollution": "None; native scale (PM mass fractions)",
    "Light": "None; native scale (lux / Kelvin)",
    "Sound": "None; native dBA (no additional log)",
    "SVF / greenery": "None; native bounded scale",
}


# Bilateral means used by H4, in addition to every source column.
AVERAGES = {
    **{f"atmo_pm{size}_avg": [f"atmo_lys_atmotube_{side}__atmotube_pm{size}" for side in ["left", "right"]]
       for size in ["1", "2.5", "10"]},
    **{f"lys_{measure}_avg": [f"atmo_lys_LYS{sensor}__lys_{measure}" for sensor in [1, 2]]
       for measure in ["lux", "medi", "kelvin"]},
}
SOUND_PROPORTION_PREFIXES = ("freq_prop_", "octave_prop_")
QC_FIELDS = {"pct_complete", "gvi_match_dist_m", "SVF_Frames_In_Window", "segment", "end_time_s", "phase_start_offset_s"}
METADATA_FIELDS = {"ParticipantID", "PhaseID", "Datetime", "Date"}

SOUND_SCREEN_SCRIPT = ROOT / "scripts/Normalization & Parameter Screening/PCA-informed_sound_frequency.py"
H4_SCRIPT = ROOT / "scripts/H4_multivariate.R"
OUT_ENVIRONMENT = OUT_DIR / "Normalization_H1toH4_environment.csv"

OUT_RESULTS = ROOT / "Presentation material/Normalization_results.md"
OUT_VOTES = OUT_DIR / "Normalization_H1toH4_transformation_votes.csv"

REFERENCE_EVIDENCE = {
    "Vocks et al 2021": "File attribution is incorrect: Knejzlikova et al. (2021), doi:10.3389/fpsyg.2021.673597. SCR used a log transform and SCL a square root; not evidence for blanket native-scale EDA.",
    "Figner_Murphy_2010": "Local full text discusses log or square-root treatment of skewed SCR and within-participant standardization; a conditional method discussion, not proof that no transformation is best.",
    "EDA_recommendations_BIOPAC": "Local 2012 recommendations discuss logarithmic or square-root SCR amplitudes; recording in microsiemens is not the same as analysing untransformed values.",
    "Laborde Mosley Thayer 2017": "Normality of HRV Parameters discusses natural-log RMSSD and absolute powers for non-normal HRV; this is general rationale, not variable-by-variable validation of our full model.",
}


# Environmental exposure treatments checked in local papers first, then primary web sources.
ENVIRONMENT_LITERATURE = [{'Group': 'Thermal',
  'ID': 'T1',
  'Paper': '[Madaniyazi et al. (2016): Outdoor Temperature, Heart Rate and Blood Pressure in Chinese Adults: '
           'Effect Modification by Individual Characteristics](https://doi.org/10.1038/srep21003)',
  'Parameters': 'Air temperature; RH',
  'Reported treatment': 'Temperature modelled with a natural cubic spline (3 df); RH with a cubic spline (4 '
                        'df).',
  'Scope / limitation': 'Nonlinear exposure-response precedent; not support for a mandatory temperature '
                        'logarithm.',
  'Checked source': 'Local PDF, p.7, Statistical analysis; publisher full text.'},
 {'Group': 'Thermal',
  'ID': 'T2',
  'Paper': '[Chinazzo et al. (2019): Daylight affects human thermal '
           'perception](https://doi.org/10.1038/s41598-019-48963-y)',
  'Parameters': 'Temperature; daylight levels',
  'Reported treatment': 'Three designed temperature levels and three daylight conditions entered as '
                        'experimental fixed factors, including their interaction.',
  'Scope / limitation': 'Factor-level experiment; not evidence for logging continuous temperature or lux.',
  'Checked source': 'Local PDF, pp.5-6, Data processing and statistical analyses.'},
 {'Group': 'Thermal',
  'ID': 'T3',
  'Paper': '[Chinazzo et al. (2020): Influence of indoor temperature and daylight illuminance on visual '
           'perception](https://doi.org/10.1177/1477153519859609)',
  'Parameters': 'Temperature; daylight levels',
  'Reported treatment': 'Three temperature and three illuminance levels analysed as experimental factors; '
                        'deviations from designed conditions included as covariates.',
  'Scope / limitation': 'Level comparisons differ from continuous within-bout exposure slopes.',
  'Checked source': 'Local PDF, pp.7-8, analysis methods; publisher record.'},
 {'Group': 'Thermal',
  'ID': 'T4',
  'Paper': '[Chinazzo et al. (2020): Effect of Indoor Temperature and Glazing with Saturated Color on Visual '
           'Perception of Daylight](https://doi.org/10.1080/15502724.2020.1726182)',
  'Parameters': 'Temperature; glazing colour',
  'Reported treatment': 'Temperature and glazing colour explicitly treated as categorical fixed factors in '
                        'mixed models.',
  'Scope / limitation': 'Supports categorical experimental coding, not logging temperature or Kelvin.',
  'Checked source': 'Local PDF, p.10, section 3.6.'},
 {'Group': 'Thermal',
  'ID': 'T5',
  'Paper': '[Rodrigues et al. (2026): The Influence of Urban Context on Emotions and Bodily Responses During '
           'Walking](https://doi.org/10.1007/s11524-025-01051-1)',
  'Parameters': 'Temperature; environmental indicators',
  'Reported treatment': 'Environmental predictors standardized before mixed-effects modelling; coefficients '
                        'interpreted per one-SD increase.',
  'Scope / limitation': 'Trip-level z-scores; does not establish our participant-phase centering rule.',
  'Checked source': 'Local PDF, pp.5-7, Statistical Analysis and coefficient interpretation.'},
 {'Group': 'Pollution',
  'ID': 'P1',
  'Paper': '[Biel et al. (2020): Acute cardiovascular health effects in a panel study of personal exposure '
           'to traffic-related air pollutants and noise in Toronto, '
           'Canada](https://doi.org/10.1038/s41598-020-73412-6)',
  'Parameters': 'UFP; black carbon; PM2.5',
  'Reported treatment': 'UFP exposure log base 5; black-carbon exposure natural log; selected HRV outcomes '
                        'separately natural-log transformed.',
  'Scope / limitation': 'Different pollutants received different treatments; UFP/BC rules cannot '
                        'automatically be transferred to PM1/PM2.5/PM10.',
  'Checked source': 'Local H4 Biel PDF and full-text Statistical analysis.'},
 {'Group': 'Pollution',
  'ID': 'P2',
  'Paper': '[Chang et al. (2015): Short-term exposure to noise, fine particulate matter and nitrogen oxides '
           'on ambulatory blood pressure: A repeated-measure '
           'study](https://doi.org/10.1016/j.envres.2015.06.004)',
  'Parameters': 'PM2.5; NOx; noise',
  'Reported treatment': 'Methods report base-10 logs of non-normally distributed continuous variables before '
                        'regression; smoothing splines also examined exposure-response form.',
  'Scope / limitation': 'Reported method, not a rule that predictor marginal normality is required.',
  'Checked source': 'Local PDF, p.2, section 2.5; publisher metadata.'},
 {'Group': 'Pollution',
  'ID': 'P3',
  'Paper': '[Shields et al. (2013): Traffic-related air pollution exposures and changes in heart rate '
           'variability in Mexico City: a panel study](https://doi.org/10.1186/1476-069X-12-7)',
  'Parameters': 'PM2.5; gaseous pollutants',
  'Reported treatment': 'Methods explicitly state no predictors were transformed; HRV outcomes used natural '
                        'logs; effects reported per IQR exposure increase.',
  'Scope / limitation': 'Direct native-predictor precedent, conditional on their model assumptions.',
  'Checked source': 'Full-text Statistical analysis; PubMed/publisher identification.'},
 {'Group': 'Pollution',
  'ID': 'P4',
  'Paper': '[Huang et al. (2014): Personal Exposure to Household Particulate Matter, Household Activities '
           'and Heart Rate Variability among Housewives](https://doi.org/10.1371/journal.pone.0089969)',
  'Parameters': 'PM2.5',
  'Reported treatment': 'PM2.5 moving averages used as exposure; effects expressed per IQR increase. HRV '
                        'outcomes were log10-transformed.',
  'Scope / limitation': 'Exposure scaling and outcome transformation are separate; no exposure logarithm '
                        'specified in the checked methods.',
  'Checked source': 'Publisher full text, Statistical analysis.'},
 {'Group': 'Pollution',
  'ID': 'P5',
  'Paper': '[Li et al. (2021): Effect of short-term exposure to particulate air pollution on heart rate '
           'variability in normal-weight and obese adults](https://doi.org/10.1186/s12940-021-00707-0)',
  'Parameters': 'PM2.5; black carbon',
  'Reported treatment': 'Moving-average pollutant exposures entered mixed models and results reported per '
                        'IQR increase; HRV and HR outcomes log10-transformed.',
  'Scope / limitation': 'No exposure log specified in the checked methods; not evidence for transforming '
                        'every PM fraction.',
  'Checked source': 'Full-text Statistical analysis and PubMed/publisher record.'},
 {'Group': 'Light',
  'ID': 'L1',
  'Paper': '[Chinazzo et al. (2019): Daylight affects human thermal '
           'perception](https://doi.org/10.1038/s41598-019-48963-y)',
  'Parameters': 'Illuminance',
  'Reported treatment': 'Designed daylight levels used as experimental fixed factors with temperature.',
  'Scope / limitation': 'Categorical dose design; not a continuous log-lux model.',
  'Checked source': 'Local PDF, pp.5-6.'},
 {'Group': 'Light',
  'ID': 'L2',
  'Paper': '[Chinazzo et al. (2020): Influence of indoor temperature and daylight illuminance on visual '
           'perception](https://doi.org/10.1177/1477153519859609)',
  'Parameters': 'Illuminance',
  'Reported treatment': 'Three daylight levels analysed with mixed models and pairwise factor-level '
                        'comparisons.',
  'Scope / limitation': 'No continuous logarithm established by this experimental design.',
  'Checked source': 'Local PDF, pp.7-8.'},
 {'Group': 'Light',
  'ID': 'L3',
  'Paper': '[Zauner et al. (2025): How to Deal With Darkness: Modeling and Visualization of Zero-Inflated '
           'Personal Light Exposure Data on a Logarithmic Scale](https://doi.org/10.1177/07487304251336624)',
  'Parameters': 'Personal light exposure; melanopic EDI',
  'Reported treatment': 'Compares logarithmic/linear/hybrid/symlog visualization and GAM approaches '
                        'including zero removal, offsets and a Tweedie error distribution.',
  'Scope / limitation': 'Light is modelled/visualized as exposure data; not a universal prescription for lux '
                        'as a physiological predictor.',
  'Checked source': 'Publisher full text and institutional full-text PDF.'},
 {'Group': 'Light',
  'ID': 'L4',
  'Paper': '[Smolders et al. (2018): Investigation of Dose-Response Relationships for Effects of White Light '
           'Exposure on Correlates of Alertness and Executive Control during Regular Daytime Working '
           'Hours](https://doi.org/10.1177/0748730418796438)',
  'Parameters': 'Illuminance',
  'Reported treatment': 'Continuous log-transformed illuminance used in dose-response analyses of alertness '
                        'and lighting appraisals.',
  'Scope / limitation': 'Direct log-illuminance predictor precedent; does not establish the optimal form for '
                        'our outcomes or for Kelvin.',
  'Checked source': 'Publisher text and author-institution full text.'},
 {'Group': 'Light',
  'ID': 'L5',
  'Paper': '[Peeters et al. (2020): What you set is (not) what you get: How a light intervention in the '
           'field translates to personal light exposure](https://doi.org/10.1016/j.buildenv.2020.107288)',
  'Parameters': 'Personal illuminance',
  'Reported treatment': 'Log-transformed illuminance aggregated in 10-minute bins and analysed using linear '
                        'mixed models.',
  'Scope / limitation': 'Illuminance is the measured response to an intervention; averaging logs differs '
                        'from logging an arithmetic mean.',
  'Checked source': 'Publisher full text, section 2.5 and Tables 4-5; institutional thesis reproducing the '
                    'paper.'},
 {'Group': 'Light',
  'ID': 'L6',
  'Paper': '[Chinazzo et al. (2020): Effect of Indoor Temperature and Glazing with Saturated Color on Visual '
           'Perception of Daylight](https://doi.org/10.1080/15502724.2020.1726182)',
  'Parameters': 'Glazing colour; visual light characteristics',
  'Reported treatment': 'Glazing colour represented by three categories in mixed models.',
  'Scope / limitation': 'Colour categories are not continuous CCT in Kelvin; no log-Kelvin rule established.',
  'Checked source': 'Local PDF, p.10, section 3.6.'},
 {'Group': 'Sound',
  'ID': 'S1',
  'Paper': '[Biel et al. (2020): Acute cardiovascular health effects in a panel study of personal exposure '
           'to traffic-related air pollutants and noise in Toronto, '
           'Canada](https://doi.org/10.1038/s41598-020-73412-6)',
  'Parameters': 'Personal noise level',
  'Reported treatment': 'Continuous noise exposure included in mixed models; selected exposure logs are '
                        'specified for UFP and BC, not noise.',
  'Scope / limitation': 'dBA-level precedent; does not validate semantic scores or spectral proportions.',
  'Checked source': 'Local H4 Biel PDF and full-text Statistical analysis.'},
 {'Group': 'Sound',
  'ID': 'S2',
  'Paper': '[Chang et al. (2015): Short-term exposure to noise, fine particulate matter and nitrogen oxides '
           'on ambulatory blood pressure: A repeated-measure '
           'study](https://doi.org/10.1016/j.envres.2015.06.004)',
  'Parameters': 'Noise level',
  'Reported treatment': 'Methods include noise among continuous variables receiving base-10 logs before '
                        'regression.',
  'Scope / limitation': 'Contrasting published treatment; not proof that an additional dBA log is needed '
                        'here.',
  'Checked source': 'Local PDF, p.2, section 2.5.'},
 {'Group': 'Sound',
  'ID': 'S3',
  'Paper': '[Kraus et al. (2013): Individual Daytime Noise Exposure during Routine Activities and Heart Rate '
           'Variability in Adults: A Repeated Measures Study](https://doi.org/10.1289/ehp.1205606)',
  'Parameters': 'A-weighted equivalent noise level',
  'Reported treatment': 'Noise modelled in dBA with piecewise linear exposure-response functions at 65 dBA; '
                        'effects per 5 dBA. HRV outcomes separately logged.',
  'Scope / limitation': 'Nonlinear function of native dBA; not additional log-dBA scaling.',
  'Checked source': 'Full-text Statistical analyses and primary abstract.'},
 {'Group': 'Sound',
  'ID': 'S4',
  'Paper': '[Sim et al. (2015): The Effects of Different Noise Types on Heart Rate Variability in '
           'Men](https://doi.org/10.3349/ymj.2015.56.1.235)',
  'Parameters': 'Noise type at controlled levels',
  'Reported treatment': 'Background, traffic, speech and mixed noise treated as exposure groups; group tests '
                        'and an adjusted general linear model.',
  'Scope / limitation': 'Sound-type contrast; does not validate transformations of YAMNet scores or '
                        'frequency proportions.',
  'Checked source': 'Local PDF, p.3, Statistical analysis.'},
 {'Group': 'Sound',
  'ID': 'S5',
  'Paper': '[Rodrigues et al. (2026): The Influence of Urban Context on Emotions and Bodily Responses During '
           'Walking](https://doi.org/10.1007/s11524-025-01051-1)',
  'Parameters': 'Noise; environmental predictors',
  'Reported treatment': 'Predictors, including the noise block, standardized before mixed-effects modelling.',
  'Scope / limitation': 'General z-score precedent; not our exact PCA or within-bout standardization method.',
  'Checked source': 'Local PDF, pp.5-7.'},
 {'Group': 'SVF / greenery',
  'ID': 'G1',
  'Paper': '[Effects of Environmental Features in Small Public Urban Green Spaces on Older Adults’ Mental '
           'Restoration: Evidence from Tokyo (2022)](https://doi.org/10.3390/ijerph19095477)',
  'Parameters': 'SVF; GVI; PET',
  'Reported treatment': 'SVF/GVI included in multilevel regression; separate curvilinear exposure-response '
                        'plots examined SVF, GVI and PET.',
  'Scope / limitation': 'No universal log/logit transformation established; perceptual restoration outcomes '
                        'differ from ours.',
  'Checked source': 'Publisher regression tables and primary PubMed figure descriptions.'},
 {'Group': 'SVF / greenery',
  'ID': 'G2',
  'Paper': '[Rodrigues et al. (2026): The Influence of Urban Context on Emotions and Bodily Responses During '
           'Walking](https://doi.org/10.1007/s11524-025-01051-1)',
  'Parameters': 'Vegetation; sky visibility; NDVI',
  'Reported treatment': 'Continuous environmental indicators standardized as z-scores for mixed-effects '
                        'analyses.',
  'Scope / limitation': 'Trip-level visual indicators differ from calibrated Holmer SVF and our within-bout '
                        'SD convention.',
  'Checked source': 'Local PDF, pp.5-7.'},
 {'Group': 'SVF / greenery',
  'ID': 'G3',
  'Paper': '[Jiang et al. (2014): A dose of nature: Tree cover, stress reduction, and gender '
           'differences](https://doi.org/10.1016/j.landurbplan.2014.08.005)',
  'Parameters': 'Visible tree-cover percentage',
  'Reported treatment': 'Tree-cover density varied continuously; a quadratic regression described the male '
                        'stress-recovery response.',
  'Scope / limitation': 'Visible-tree dose precedent, not a Holmer SVF transformation study.',
  'Checked source': 'US Forest Service author full text and primary abstract.'},
 {'Group': 'SVF / greenery',
  'ID': 'G4',
  'Paper': '[Jiang et al. (2016): A Dose-Response Curve Describing the Relationship Between Urban Tree Cover '
           'Density and Self-Reported Stress Recovery](https://doi.org/10.1177/0013916514552321)',
  'Parameters': 'Visible tree-cover percentage',
  'Reported treatment': 'Primary abstract reports a positive linear relationship between tree-cover density '
                        'and self-reported recovery, adjusted for covariates.',
  'Scope / limitation': 'Primary abstract checked; detailed preprocessing/logging cannot be established from '
                        'the abstract.',
  'Checked source': 'Author-institution repository abstract; not full-method verification.'},
 {'Group': 'SVF / greenery',
  'ID': 'G5',
  'Paper': '[Assessing the restorative benefits of open rooftop green spaces characteristics using virtual '
           'reality and EEG analysis (2026)](https://doi.org/10.3389/fpsyg.2026.1807425)',
  'Parameters': 'GVI; SVF',
  'Reported treatment': 'GVI and SVF divided into low/medium/high design levels in virtual-reality '
                        'scenarios.',
  'Scope / limitation': 'Categorical scenario design; not a continuous SVF logit or native-linear '
                        'validation.',
  'Checked source': 'Publisher full text, sections 2 and 3.1.'},
 {'Group': 'Pollution',
  'ID': 'P6',
  'Paper': '[Contribution of Long-Term Exposure to Outdoor Black Carbon to the Carcinogenicity of Air '
           'Pollution: Evidence regarding Risk of Cancer in the Gazel '
           'Cohort](https://doi.org/10.1289/EHP8719)',
  'Parameters': 'Black carbon; PM2.5',
  'Reported treatment': 'Natural-log exposure predictors in Cox models',
  'Scope / limitation': 'Long-term cumulative exposure and cancer; not short-term wearable physiology.',
  'Checked source': 'Primary full-text statistical methods; PMC7989243.'},
 {'Group': 'Pollution',
  'ID': 'P7',
  'Paper': '[Association between personal exposure to household air pollution and gestational blood pressure '
           'among women using solid cooking fuels in rural Tamil Nadu, '
           'India](https://doi.org/10.1016/j.envres.2022.112756)',
  'Parameters': 'PM2.5; black carbon; CO',
  'Reported treatment': 'Natural-log exposure predictors; functional form assessed with partial residual '
                        'plots',
  'Scope / limitation': 'Pregnancy and household exposure; quartile and smooth-model alternatives also '
                        'examined.',
  'Checked source': 'Primary full-text methods; PMC8935388.'},
 {'Group': 'Pollution',
  'ID': 'P8',
  'Paper': '[Long-term Exposure to Black Carbon and Carotid Intima-Media Thickness: The Normative Aging '
           'Study](https://doi.org/10.1289/ehp.1104845)',
  'Parameters': 'Black carbon',
  'Reported treatment': 'Natural-log black-carbon predictor in primary mixed models',
  'Scope / limitation': 'Annual exposure in older men; CIMT outcome also logged separately. Not PM '
                        'mass-fraction evidence.',
  'Checked source': 'Primary author manuscript, statistical methods; PMC3764069.'},
 {'Group': 'Pollution',
  'ID': 'P9',
  'Paper': '[Concentration?Response Functions Relating Concentration and Duration of Ambient Traffic-Related '
           'Air Pollution Exposure to Systolic Blood Pressure](https://doi.org/10.1289/EHP13854)',
  'Parameters': 'Black carbon; particle number concentration',
  'Reported treatment': 'Natural-log concentration and duration predictors in concentration-response '
                        'functions',
  'Scope / limitation': 'Particle number is related to UFP, not interchangeable with PM2.5 mass.',
  'Checked source': 'Primary published concentration-response equations; PMC11068155.'},
 {'Group': 'Pollution',
  'ID': 'P10',
  'Paper': '[Baseline exposure-response assessment of metals from dried blood spots and personal household '
           'air pollution concentrations among pregnant women in Rwanda: household air pollution '
           'intervention network trial](https://doi.org/10.1186/s41043-025-01178-6)',
  'Parameters': 'PM2.5; black carbon; CO',
  'Reported treatment': 'Natural-log 24-hour exposure predictors',
  'Scope / limitation': 'Pregnancy blood-metal outcomes; explicit abstract methods checked, full text not '
                        'retrieved.',
  'Checked source': 'Primary PubMed abstract, PMID41420186.'},
 {'Group': 'Pollution',
  'ID': 'P11',
  'Paper': '[Relationship of Time-Activity-Adjusted Particle Number Concentration with Blood '
           'Pressure](https://doi.org/10.3390/ijerph15092036)',
  'Parameters': 'Time-activity-adjusted particle number concentration',
  'Reported treatment': 'Natural-log exposure predictor',
  'Scope / limitation': 'Particle-number proxy for UFP; not PM2.5 mass or log-base-5 precedent.',
  'Checked source': 'Primary publisher methods, PMC6165221.'},
 {'Group': 'Pollution',
  'ID': 'P12',
  'Paper': '[Respiratory outcomes of ultrafine particulate matter (UFPM) as a surrogate measure of '
           'near-roadway exposures among bicyclists](https://doi.org/10.1186/s12940-017-0212-x)',
  'Parameters': 'Ultrafine particle concentration',
  'Reported treatment': 'Natural-log median exposure predictor in mixed models',
  'Scope / limitation': 'Cyclist lung-function response; not the same outcome or time scale as H4.',
  'Checked source': 'Primary full-text statistical methods, PMC5299642.'},
 {'Group': 'Light',
  'ID': 'L7',
  'Paper': '[Melanopic irradiance defines the impact of evening display light on sleep latency, melatonin '
           'and alertness](https://doi.org/10.1038/s42003-023-04598-4)',
  'Parameters': 'Melanopic EDI; photopic illuminance',
  'Reported treatment': 'Log-base-10 exposure-response predictors',
  'Scope / limitation': 'Evening display exposure; melanopic and photopic metrics are distinguished from '
                        'Kelvin.',
  'Checked source': 'Primary published dose-response methods and tables, PMC9974389.'},
 {'Group': 'Light',
  'ID': 'L8',
  'Paper': '[A higher illuminance reduces momentary exhaustion in exhausted employees: results from a field '
           'study](https://doi.org/10.1016/j.jenvp.2025.102543)',
  'Parameters': 'Hourly personal illuminance',
  'Reported treatment': 'Log illuminance; person-mean-centered hourly exposure and grand-mean-centered '
                        'person mean in multilevel models',
  'Scope / limitation': 'Separates within- and between-person light exposure; not participant-phase '
                        'centering or the exact H4 scaling rule. Log base not established here.',
  'Checked source': 'Author-hosted published PDF, methods p5 and model tables p7?8; '
                    'https://pure.tue.nl/ws/portalfiles/portal/355235424/1-s2.0-S027249442500026X-main.pdf'},
 {'Group': 'Pollution',
  'ID': 'P13',
  'Paper': '[Exposure to mobile source air pollution in early life and childhood asthma incidence: The '
           'Kaiser Air Pollution and Pediatric Asthma Study](https://doi.org/10.1097/EDE.0000000000000754)',
  'Parameters': 'PM2.5; NOx; CO',
  'Reported treatment': 'Natural-log exposure predictors',
  'Scope / limitation': 'Long-term early-life mobile-source pollution; childhood asthma, not ten-second '
                        'physiology.',
  'Checked source': 'Primary author manuscript / CDC PDF, PMC5718963; exposure-response results per '
                    'natural-log increase.'},
 {'Group': 'Pollution',
  'ID': 'P14',
  'Paper': '[Association between early life constituent-specific PM2.5 exposure and child development and '
           'the moderating role of greenness: a nationwide study in '
           'China](https://doi.org/10.1186/s12916-026-04750-x)',
  'Parameters': 'PM2.5 and constituents, including black carbon',
  'Reported treatment': 'Natural-log exposure predictors before regression',
  'Scope / limitation': 'Developmental outcomes and long-term early-life exposure; not the H4 '
                        'repeated-measures design.',
  'Checked source': 'Primary published statistical methods / results; PMC13077890; concentrations explicitly '
                    'natural-log transformed before regression.'},
 {'Group': 'Light',
  'ID': 'L9',
  'Paper': '[Evening home lighting adversely impacts the circadian system and '
           'sleep](https://doi.org/10.1038/s41598-020-75622-4)',
  'Parameters': 'Melanopic illuminance',
  'Reported treatment': 'Log exposure predictors: individual mean and deviation from individual mean',
  'Scope / limitation': 'Sleep wakefulness GLMM; melanopic rather than photopic illuminance; log base not '
                        'established here.',
  'Checked source': 'Primary publisher Methods, Sleep variables; within-person mean and deviation of '
                    'log-transformed melanopic illuminance.'},
 {'Group': 'Light',
  'ID': 'L10',
  'Paper': '[Associations between light exposure and sleep timing and sleepiness while awake in a sample of '
           'UK adults in everyday life](https://doi.org/10.1073/pnas.2301608120)',
  'Parameters': 'Melanopic EDI; photopic illuminance',
  'Reported treatment': 'Log-base-10 lux in statistical analyses; log melanopic exposure/history predictors '
                        'in mixed models',
  'Scope / limitation': 'Field sleep and sleepiness outcomes; different light metric and time scale from H4.',
  'Checked source': 'Primary PNAS Statistical Analysis: log10 lux, acute exposure and '
                    '30-minute-to-eight-hour light histories.'},
 {'Group': 'Thermal',
  'ID': 'T6',
  'Paper': '[Effects of Indoor Thermal Environment on Human Food Intake, Productivity, and Comfort: Pilot, '
           'Randomized, Crossover Trial](https://doi.org/10.1002/oby.22328)',
  'Parameters': 'Air temperature conditions',
  'Reported treatment': 'Categorical experimental thermal conditions',
  'Scope / limitation': '25 women; 19?20 versus 26?27 deg C, randomized crossover; not continuous '
                        'exposure-response inference.',
  'Checked source': 'Primary PubMed abstract PMID30426700 explicitly identifies conditions and mixed-effects '
                    'thermal-condition comparison.'},
 {'Group': 'Thermal',
  'ID': 'T7',
  'Paper': '[The Effects of High-Temperature Weather on Human Sleep Quality and '
           'Appetite](https://doi.org/10.3390/ijerph16020270)',
  'Parameters': 'Daily maximum temperature conditions',
  'Reported treatment': 'Comparisons across four experimental temperature levels',
  'Scope / limitation': 'Ten students; 28,32,36,38 deg C; a controlled-level comparison, not continuous H4 '
                        'temperature or heat-index scaling.',
  'Checked source': 'Primary publisher and PubMed PMID30669302: four temperature conditions; categorical '
                    'condition design confirmed.'},
 {'Group': 'Sound',
  'ID': 'S6',
  'Paper': '[Effects of Aircraft Noise Exposure on Heart Rate during Sleep in the Population Living Near '
           'Airports](https://doi.org/10.3390/ijerph16020269)',
  'Parameters': 'A-weighted sound levels: LAeq, LA90 and LAmax',
  'Reported treatment': 'Continuous dBA-scale predictors; effects per 10 dBA, no extra exposure log '
                        'specified',
  'Scope / limitation': 'Sleep and aircraft noise; 15-second energy indicators and separate event models, '
                        'not source-category scores.',
  'Checked source': 'Primary full text section2.4 and coefficient tables; PMC6352139.'},
 {'Group': 'Sound',
  'ID': 'S7',
  'Paper': '[Personal Exposure to Household Particulate Matter, Household Activities and Heart Rate '
           'Variability among Housewives](https://doi.org/10.1371/journal.pone.0089969)',
  'Parameters': 'Noise level',
  'Reported treatment': 'Noise dBA covariate in mixed-effects models; additional noise log not specified',
  'Scope / limitation': 'HRV outcome log10 does not imply log10 noise. Same publication as P4, counted once '
                        'within Sound.',
  'Checked source': 'Primary full text statistical methods and Table3; PMC3940710.'},
 {'Group': 'Sound',
  'ID': 'S8',
  'Paper': '[Short-term association between personal exposure to noise and heart rate variability: The '
           'RECORD MultiSensor Study](https://doi.org/10.1016/j.envpol.2017.08.031)',
  'Parameters': 'Personal noise Leq',
  'Reported treatment': 'Continuous dBA and piecewise dBA with 65 dBA breakpoint',
  'Scope / limitation': 'Five-minute windows, 75 participants; mixed effects and temporal autocorrelation; '
                        'HRV logged separately.',
  'Checked source': 'Primary publisher abstract and Results: per-one-dBA effects and piecewise regressions.'}]

# Count only the parameter treatment actually identified in each checked source.
ENVIRONMENT_TREATMENT_COUNTS = [('Thermal', 'Air temperature; RH', 'Spline exposure-response', ['T1']),
 ('Thermal', 'Air temperature', 'Categorical experimental levels', ['T2', 'T3', 'T4', 'T6', 'T7']),
 ('Thermal', 'Temperature/environmental predictors', 'Z-score scaling', ['T5']),
 ('Pollution', 'UFP', 'Log base 5', ['P1']),
 ('Pollution', 'Black carbon', 'Natural log', ['P1', 'P6', 'P7', 'P8', 'P9', 'P10', 'P14']),
 ('Pollution', 'PM2.5; NOx', 'Log base 10', ['P2']),
 ('Pollution', 'PM2.5 / pollutant predictors', 'No transformation explicitly stated', ['P3']),
 ('Pollution', 'PM2.5', 'Native-unit / IQR exposure reporting; exposure log not specified', ['P4', 'P5']),
 ('Light', 'Illuminance', 'Categorical experimental levels', ['L1', 'L2']),
 ('Light',
  'Personal light exposure / melanopic EDI',
  'Comparison of log/zero-handling methods; not one adopted predictor transformation',
  ['L3']),
 ('Light',
  'Illuminance as predictor',
  'Log transformation (base not established in this review)',
  ['L4', 'L8']),
 ('Light',
  'Illuminance as measured response',
  'Log transformation (not a physiological-predictor precedent)',
  ['L5']),
 ('Light', 'Glazing colour', 'Categorical experimental levels; not continuous Kelvin', ['L6']),
 ('Sound', 'Noise level', 'Continuous dBA; additional exposure log not specified', ['S1']),
 ('Sound', 'Noise level', 'Log base 10', ['S2']),
 ('Sound', 'Noise level', 'Piecewise linear dBA', ['S3']),
 ('Sound', 'Noise type', 'Categorical groups', ['S4']),
 ('Sound', 'Noise/environmental predictors', 'Z-score scaling', ['S5']),
 ('SVF / greenery',
  'SVF; GVI',
  'Multilevel regression / separate curvilinear assessment; log/logit not specified',
  ['G1']),
 ('SVF / greenery', 'Vegetation; sky visibility; NDVI', 'Z-score scaling', ['G2']),
 ('SVF / greenery', 'Tree-cover percentage', 'Quadratic exposure-response', ['G3']),
 ('SVF / greenery',
  'Tree-cover percentage',
  'Linear association in primary abstract; preprocessing not verified',
  ['G4']),
 ('SVF / greenery', 'SVF; GVI', 'Categorical low/medium/high levels', ['G5']),
 ('Pollution', 'PM2.5 as health predictor', 'Natural log', ['P6', 'P7', 'P10', 'P13', 'P14']),
 ('Pollution',
  'UFP / particle-number exposure as health predictor',
  'Natural log; related particle-count metrics',
  ['P9', 'P11', 'P12']),
 ('Light', 'Melanopic EDI / photopic illuminance as predictor', 'Log base 10', ['L7']),
 ('Light',
  'Light exposure as health predictor (photopic and melanopic metrics)',
  'Logarithmic predictor; exact bases and metrics differ by paper',
  ['L4', 'L7', 'L8', 'L9', 'L10']),
 ('Sound',
  'Noise level as predictor / covariate',
  'dBA-scale exposure without additional log specified; includes piecewise forms',
  ['S1', 'S3', 'S6', 'S7', 'S8'])]


In [3]:
# 2. Load
raw = pd.read_csv(DATA, low_memory=False)
missing_cols = [c for c in COL.values() if c not in raw.columns]
assert not missing_cols, missing_cols

paper_files = tuple(p for p in PAPERS.rglob("*") if p.suffix.lower() == ".pdf")
paper_paths = {}
for key in dict.fromkeys(key for options in USED.values() for _, keys in options for key in keys):
    hits = [p for p in paper_files if key in p.name]
    if len(hits) != 1:
        raise FileNotFoundError(f"{key} -> {[str(p) for p in hits]}")
    paper_paths[key] = hits[0]
core_values = raw.loc[raw.PhaseID.isin(CORE_PHASES), list(COL.values())].apply(pd.to_numeric, errors="coerce")
nonpositive_counts = core_values.le(0).sum()

source_columns = list(raw.columns)
for name, sources in AVERAGES.items():
    if not set(sources) <= set(raw):
        raise ValueError(f"Missing bilateral sources for {name}: {sources}")
    raw[name] = raw[sources].mean(axis=1)
core = raw.loc[raw.PhaseID.isin(CORE_PHASES)].copy()

# Import declared PCA inputs without running the screening workflow.
spec = importlib.util.spec_from_file_location("sound_screen_definition", SOUND_SCREEN_SCRIPT)
sound_screen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sound_screen)
sound_pca_columns = tuple(sound_screen.PCA_COLUMNS)
eq1_line = next(line for line in H4_SCRIPT.read_text(encoding="utf-8").splitlines() if line.strip().startswith("Y ~"))
eq1_terms = set(re.findall(r"[A-Za-z][A-Za-z0-9_.]*", eq1_line))


C:\Users\pandya\AppData\Roaming\Python\Python39\site-packages\scipy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\pandya\AppData\Roaming\Python\Python39\site-packages\scipy\.libs\libopenblas.PZA5WNOTOH6FZLB2KBVKAURAKVTFSNNU.gfortran-win_amd64.dll
C:\Users\pandya\AppData\Roaming\Python\Python39\site-packages\scipy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [4]:
# 3. Define
def step5_text(col_name, transformation):
    return "Not in Step 5" if col_name in NOT_IN_STEP5 else STEP5_TEXT[transformation]

def check_log_domain(col_name, transformation):
    return int(nonpositive_counts[col_name]) if transformation == "Natural log" else 0


def make_normalization_tables():
    catalog = []
    number_of = {}
    rows = []
    physio_rows = []
    for parameter, options in USED.items():
        transformation, keys = options[0]
        marks, titles = [], []
        for key in keys:
            path = paper_paths[key]
            if path.name not in number_of:
                number_of[path.name] = len(catalog) + 1
                catalog.append({"Number": f"[{number_of[path.name]}]", "Paper title": path.name, "File path": str(path), "Evidence checked": REFERENCE_EVIDENCE.get(key, "Existing mapping; not newly full-text verified")})
            marks.append(f"[{number_of[path.name]}]")
            titles.append(path.name)
        col_name = COL[parameter]
        rows.append({
            "col_name": col_name,
            "Parameter": parameter,
            "Transformation": transformation,
            "Number of papers": len(marks),
            "Papers": ", ".join(marks),
        })
        physio_rows.append({
            "Column": col_name,
            "Parameter": parameter,
            "Unit": UNIT[col_name],
            "Transformation in the papers": ("Mixed / conditional: SCR log or square root; SCL square root in the checked 2021 paper" if "conductance" in parameter.lower() else "Natural-log HRV rationale; variable-specific mapping not fully verified" if transformation == "Natural log" else "Existing mapped references; native-scale optimum not established"),
            "Transformation in Step 5": step5_text(col_name, transformation),
            "Rows excluded (x <= 0)": check_log_domain(col_name, transformation),
            "Paper titles": " | ".join(titles),
        })
    
    return pd.DataFrame(rows), pd.DataFrame(catalog), pd.DataFrame(physio_rows)


def transformation_policy(column, series):
    if column in COL.values():
        parameter = next(p for p, c in COL.items() if c == column)
        transformation = USED[parameter][0][0]
        return "Physiological outcome", STEP5_TEXT[transformation], "Existing literature mapping; transform before model-specific standardization"
    if column in METADATA_FIELDS or column.endswith("_Date") or column.startswith("ucm_GPS_") or column == "ucm_MAG_hdg":
        return "Metadata", "Not applicable", "Identifier, timestamp or positioning metadata; not a continuous exposure transformation"
    if column in QC_FIELDS or column.startswith("empatica_hrv_fd_5min_") or column.startswith("SVF_") and column != "SVF":
        return "QC / processing metadata", "Not applicable", "Availability, quality or processing descriptor; retained for checking"
    if column == "unidentify_prop":
        return "Classification diagnostic", "Not applicable to exposure model", "Unidentified sound share; excluded before sound PCA"
    if column == "category" or column.endswith(("_SR", "_lux_cat", "__lux_cat")) or column == "ucm_IO_flag":
        return "Categorical / ordinal code", "No continuous transformation", "Use declared category coding; numerical storage does not imply a continuous measurement"
    if column.startswith("questionnaire_"):
        score = any(x in column for x in ["_mean_", "_comfort_", "_pleasantness_", "_familiarity_"])
        return ("Questionnaire score" if score else "Questionnaire / participant metadata", "Native score or categorical coding; no automatic log", "Repeated questionnaire values do not become independent ten-second observations; model depends on item/score type")
    if column.endswith("_%") or column in {"SVF", "gvi_50m"} or column.startswith(("gvi_", "tree_phase_", "grass_phase_", "plant_phase_")):
        return "SVF / greenery exposure", "Native scale before model-specific centering/scaling", "Bounded exposure; no automatic log/logit. Boundary values and model role must be checked"
    if column == "ucm_SND_dBA" or column.startswith(SOUND_PROPORTION_PREFIXES) or column.endswith(("_score", "_prop")) or column in {"dominant_frequency_hz", "spectral_centroid_hz", "spectral_rolloff85_hz"}:
        return "Sound exposure", "Native scale; within-bout centering and unit-SD scaling for sound PCA", "dBA is already logarithmic; do not log again. Proportions/scores are not probabilities of equivalent meaning; current PCA is correlation-based"
    if pd.api.types.is_numeric_dtype(series):
        return "Numeric measurement / derived field", "Native scale before model-specific centering/scaling", "No additional nonlinear transform introduced. Predictor normality alone is not a model requirement; outcome and residual assumptions need separate checks"
    return "Text / categorical field", "No continuous transformation", "Preserve labels; no z-score or numeric log of text"


def make_column_inventory(physio_counts):
    existing = physio_counts.set_index("col_name")
    rows = []
    for column in raw.columns:
        values = core[column]
        role, treatment, basis = transformation_policy(column, values)
        numeric = pd.to_numeric(values, errors="coerce") if pd.api.types.is_numeric_dtype(values) else pd.Series(dtype=float)
        valid = numeric.dropna()
        ref = existing.loc[column].to_dict() if column in existing.index else {}
        rows.append({"col_name": column, "Parameter": ref.get("Parameter", column),
            "Role": role, "Transformation": treatment, "Basis": basis,
            "Scaling": "Model-specific; metadata/QC/categorical fields are not automatically standardized",
            "Number of papers": ref.get("Number of papers", 0), "Papers": ref.get("Papers", ""),
            "Origin": "Bilateral mean" if column in AVERAGES else "Merged source",
            "Core rows": len(core), "Missing": int(values.isna().sum()), "Unique": values.nunique(),
            "Nonpositive": int((valid <= 0).sum()) if len(valid) else None,
            "Minimum": valid.min(), "Maximum": valid.max(),
            "Skewness": valid.skew() if len(valid) >= 3 and valid.nunique() > 1 else None,
            "Nonfinite": int((~np.isfinite(numeric.dropna())).sum()) if len(numeric) else 0})
    table = pd.DataFrame(rows)
    assert not table.col_name.duplicated().any()
    assert set(source_columns) <= set(table.col_name)
    assert len(table) == len(source_columns) + len(AVERAGES)
    return table


def make_paper_roles():
    rows = []
    for parameter, options in USED.items():
        transformation, keys = options[0]
        for key in keys:
            path = paper_paths[key]
            rows.append({"folder": str(path.parent.relative_to(PAPERS)), "file": path.name,
                         "role": "Physiological transformation reference",
                         "use": REFERENCE_EVIDENCE.get(key, f"{COL[parameter]}: mapped reference for current {transformation} policy; endorsement not newly verified")})
    return pd.DataFrame(rows)



def _unique_env_ids(ids):
    return sorted(set(ids), key=lambda i: (i[0], int(i[1:]) if i[1:].isdigit() else 999, i))


def make_physio_vote_tables():
    """Candidate transforms, paper counts, and literature winner per shared-parameter group."""
    detail_rows = []
    summary_rows = []
    for block in PHYSIO_LITERATURE_VOTES:
        params = block["parameters"]
        step5 = STEP5_TEXT[USED[params[0]][0][0]]
        param_str = "; ".join(params)
        scored = [(label, len(keys), keys) for label, keys in block["candidates"]]
        max_count = max(s[1] for s in scored)
        winners = [s[0] for s in scored if s[1] == max_count]
        lit_winner = winners[0] if len(winners) == 1 else "Tie: " + "; ".join(winners)
        aligned = step5 in lit_winner or (lit_winner.startswith("No") and step5 == "No transformation")
        if block["vote_group"].startswith("EDA"):
            aligned = False
        summary_rows.append({
            "Vote group": block["vote_group"],
            "Parameters (one shared rule)": param_str,
            "Literature winner": lit_winner,
            "Winning paper count": max_count,
            "Step 5 transformation": step5,
            "Aligned with literature winner": "Yes" if aligned else "No — model contract retained",
        })
        for label, keys in block["candidates"]:
            detail_rows.append({
                "Vote group": block["vote_group"],
                "Parameters": param_str,
                "Candidate transformation": label,
                "Paper count": len(keys),
                "Paper keys": ", ".join(keys),
                "Is literature winner": label in winners and len(winners) == 1,
            })
    detail = pd.DataFrame(detail_rows)
    summary = pd.DataFrame(summary_rows)
    assert set(summary["Vote group"]) == {b["vote_group"] for b in PHYSIO_LITERATURE_VOTES}
    return summary, detail


def make_environment_vote_tables():
    """Roll up checked environmental papers by group and candidate nonlinear treatment."""
    from collections import defaultdict
    detail_rows = []
    summary_rows = []
    for group in ["Thermal", "Pollution", "Light", "Sound", "SVF / greenery"]:
        buckets = defaultdict(list)
        for g, _param, treatment, ids in ENVIRONMENT_TREATMENT_COUNTS:
            if g != group:
                continue
            buckets[treatment].extend(ids)
        ranked = sorted(buckets.items(), key=lambda x: (-len(set(x[1])), x[0]))
        if not ranked:
            continue
        top_count = len(set(ranked[0][1]))
        top_treatments = [t for t, ids in ranked if len(set(ids)) == top_count]
        lit_winner = top_treatments[0] if len(top_treatments) == 1 else "Tie: " + "; ".join(top_treatments[:3])
        h4 = ENV_H4_NONLINEAR[group]
        summary_rows.append({
            "Exposure group": group,
            "Literature winner (nonlinear)": lit_winner,
            "Winning paper count": top_count,
            "Current H4 nonlinear rule": h4,
            "Aligned": "Partial — group-level; see per-parameter mapping",
        })
        for treatment, ids in ranked:
            uid = _unique_env_ids(ids)
            detail_rows.append({
                "Exposure group": group,
                "Candidate treatment": treatment,
                "Paper count": len(uid),
                "Paper IDs": ", ".join(uid),
                "Is literature winner": treatment in top_treatments and len(top_treatments) == 1,
            })
    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def make_transformation_vote_table():
    physio_summary, physio_detail = make_physio_vote_tables()
    env_summary, env_detail = make_environment_vote_tables()
    return physio_summary, physio_detail, env_summary, env_detail


def make_environment_table(inventory):
    sensor = inventory.col_name.str.startswith(("ucm_", "atmo_lys_", "atmo_pm", "lys_"))
    exposure = inventory.Role.isin(["Sound exposure", "SVF / greenery exposure", "Classification diagnostic"])
    selected = inventory.loc[(sensor | exposure) & ~inventory.Role.isin(["Metadata", "QC / processing metadata"])].copy()
    rows = []
    for _, row in selected.iterrows():
        column = row.col_name
        diagnostic = row.Role == "Classification diagnostic"
        categorical = row.Role == "Categorical / ordinal code"
        sound = column in sound_pca_columns
        nonlinear = "Not applicable" if diagnostic or categorical else "None; native scale"
        scaling = "No continuous scaling specified" if diagnostic or categorical else "Within-bout centering and pooled-SD scaling when entered in H4"
        if sound:
            scaling += "; same centering/scaling in sound PCA"
        reason = "Current specification; model form and residual assumptions require separate checks"
        if column == "ucm_SND_dBA":
            reason = "Already logarithmic dBA; no additional log applied"
        elif diagnostic:
            reason = "Classification review only; excluded before sound PCA"
        elif categorical:
            reason = "Use category coding; numeric codes are not continuous measurements"
        elif column.endswith(("_%", "_prop")) or column.startswith(SOUND_PROPORTION_PREFIXES) or column == "SVF":
            reason = "Bounded exposure; native-scale zeros allowed; no automatic log/logit"
        values = pd.to_numeric(core[column], errors="coerce")
        values = values[np.isfinite(values)]
        log_candidate = column.startswith("atmo_pm") or ("atmotube_pm" in column) or column in {"lys_lux_avg", "lys_medi_avg"} or column.endswith(("__lys_lux", "__lys_medi"))
        positive = values[values > 0]
        log_skew = np.log(positive).skew() if log_candidate and len(positive) >= 3 else np.nan
        alternative = "ln(x) model-form sensitivity; not applied" if log_candidate else "No additional nonlinear transform proposed automatically"
        evidence = "Native scale is the current specification, not an optimized or universally validated choice"
        if "lux" in column or "medi" in column:
            evidence = "Zauner et al. 2025, doi:10.1177/07487304251336624: zero handling for logarithmic light modelling; does not mandate logging every light predictor"
        elif "pm" in column:
            evidence = "Biel et al. 2020, doi:10.1038/s41598-020-73412-6: pollutant-specific transforms (UFP/BC), not a universal PM transform"
        rows.append({"Column": column, "Role": row.Role, "Nonlinear transformation": nonlinear,
            "Centering / scaling": scaling, "Condition / reason": reason,
            "Sound PCA input": sound, "Current Eq1 term": column in eq1_terms,
            "Specification": str(SOUND_SCREEN_SCRIPT if sound else H4_SCRIPT),
            "Finite observations": len(values), "Nonpositive observations": int((values <= 0).sum()),
            "Native skewness": values.skew() if len(values) >= 3 and values.nunique() > 1 else np.nan,
            "Positive-only log skewness": log_skew, "Candidate alternative": alternative,
            "Evidence scope": evidence})
    table = pd.DataFrame(rows)
    assert table.Column.is_unique
    assert set(table.loc[table["Sound PCA input"], "Column"]) == set(sound_pca_columns)
    return table


def make_normalization_report(inventory, physio, environment, physio_vote_summary, physio_vote_detail, env_vote_summary, env_vote_detail):
    assert len(physio_vote_summary) == len(PHYSIO_LITERATURE_VOTES)
    physiological = physio.drop(columns=["Rows excluded (x <= 0)"]).merge(
        inventory[["col_name", "Number of papers", "Papers"]].rename(columns={"col_name": "Column"}),
        on="Column", how="left", validate="one_to_one")
    assert len(physiological) == 27 and physiological["Number of papers"].notna().all()
    physiological["Number of papers"] = physiological["Number of papers"].astype(int)
    physiological["Unit"] = physiological["Unit"].replace({"�C": "deg C"})
    physiological = physiological[["Parameter", "Column", "Unit", "Transformation in the papers",
        "Transformation in Step 5", "Number of papers", "Papers", "Paper titles"]]
    environmental = environment[["Column", "Nonlinear transformation", "Centering / scaling",
        "Condition / reason", "Candidate alternative"]]
    evidence = pd.DataFrame(ENVIRONMENT_LITERATURE)
    group_counts = evidence.groupby("Group", sort=False).size().rename("Papers reviewed").reset_index()
    assert group_counts["Papers reviewed"].ge(5).all()
    summaries = pd.DataFrame([
        {"Group": "Thermal", "Current H4": "Native units, then within-bout centering / pooled-SD scaling for entered terms", "Literature finding": "Temperature splines, experimental levels and z-scores are all used. These papers do not establish one rule for every heat index, wind or radiation channel."},
        {"Group": "Pollution", "Current H4": "Native PM, then within-bout centering / pooled-SD scaling", "Literature finding": "Native PM exposure / IQR reporting and selected pollutant logs both occur; logging an HRV outcome is not logging its PM predictor."},
        {"Group": "Light", "Current H4": "Native lux and Kelvin, then within-bout centering / pooled-SD scaling", "Literature finding": "Continuous log-lux and categorical illuminance designs both occur. Evidence for log-lux does not establish log-Kelvin or log-RGB transformations."},
        {"Group": "Sound", "Current H4": "dBA and selected sound descriptors centered within bout and pooled-SD scaled", "Literature finding": "Studies use dBA, piecewise dBA, log10 noise, standardized exposures or sound-type groups. These are not validations of all YAMNet / frequency transformations."},
        {"Group": "SVF / greenery", "Current H4": "Native bounded exposures, then within-bout centering / pooled-SD scaling", "Literature finding": "Published models use standardized visual indicators, linear or quadratic tree cover, and categorical SVF/GVI scenarios; no universal logit rule follows."}
    ]).merge(group_counts, on="Group", validate="one_to_one")
    treatment_counts = pd.DataFrame([
        {"Group": group, "Parameter counted": parameter, "Treatment in the papers": treatment,
         "Number of papers": len(ids), "Papers": ", ".join(ids)}
        for group, parameter, treatment, ids in ENVIRONMENT_TREATMENT_COUNTS])
    assert set(", ".join(treatment_counts["Papers"]).split(", ")) == set(evidence["ID"])
    evidence_sections = []
    for group, rows in evidence.groupby("Group", sort=False):
        evidence_sections += ["**" + group + ": checked papers**", "",
            rows.drop(columns=["Group"]).to_markdown(index=False), ""]
    return "\n".join([
        "# Normalization results", "",
        "**Physiological transformations and paper counts**", "",
        "Paper counts are mapped references, not independently verified counts of papers using the selected transformation.", "",
        physiological.to_markdown(index=False), "",
        "**Literature-based transformation voting (physiology)**", "",
        "Vote groups tie related parameters to one Step 5 transformation. Paper counts use mapped local PDF keys only (see also `Presentation material/Reference Literature for all Paper3.md`). A candidate needs the highest count within its group; five papers support the winning candidate where noted below.", "",
        physio_vote_summary.to_markdown(index=False), "",
        physio_vote_detail.to_markdown(index=False), "",
        "**Literature-based transformation voting (environment)**", "",
        "Nonlinear exposure treatments from checked papers (`ENVIRONMENT_LITERATURE`). Centering/scaling is separate. Current H4 uses native scale for entered continuous channels unless noted in the complete mapping table.", "",
        env_vote_summary.to_markdown(index=False), "",
        env_vote_detail.to_markdown(index=False), "",
        "**Environmental transformations**", "",
        treatment_counts[treatment_counts["Number of papers"].ge(5)].to_markdown(index=False), "",
        "**Additional exact treatments and evidence gaps**", "",
        treatment_counts[treatment_counts["Number of papers"].lt(5)].to_markdown(index=False), "",
        "Counts refer to the treatment of the named environmental parameter, not to transformations of HRV or other outcomes. Z-score scaling, exposure-response shape, categorical coding and logarithms are kept separate. A paper may contribute to more than one parameter row; counts must not be summed as unique papers. IDs link these counts to the paper-by-paper tables below.", "",
        "**Current environmental treatment and literature interpretation**", "",
        summaries[["Group", "Papers reviewed", "Literature finding", "Current H4"]].to_markdown(index=False), "",
        "These are group-level reviewed-paper counts, not votes for one transformation or five exact-method endorsements for every raw channel. Full-text methods were checked where available; the Jiang 2016 entry is explicitly abstract-only. Current H4 transformations are unchanged. Five-paper support is now documented for temperature condition comparisons, natural-log PM2.5, natural-log black carbon, logarithmic light predictors and dBA-scale noise predictors. Exact log bases, photopic versus melanopic light, and linear versus piecewise dBA are identified per paper; these are not five replications of the full H4 method. Other exact treatments remain below five and are listed separately. Black-carbon and particle-number studies are related evidence, not substitutes for PM1/PM2.5/PM10 evidence.", "",
        *evidence_sections,
        "**Complete environmental parameter mapping**", "",
        environmental.to_markdown(index=False), "",
        "**Method and literature**", "",
        "Distinguish (1) nonlinear changes such as ln(x), (2) within-bout centering/scaling, and (3) role/categorical exclusions. [NIST](https://www.itl.nist.gov/div898/handbook/pmd/section6/pmd624.htm) relates transformations to regression form and error variance. Predictor skewness is a diagnostic, not proof that a transform is required or sufficient. [Enders & Tofighi (2007)](https://doi.org/10.1037/1082-989X.12.2.121) explain group-mean centering; the participant-phase grouping and pooled-SD scaling are this study's specifications.", "",
        "**Paper index**", "",
        papers_table[["Number", "Paper title"]].to_markdown(index=False), "",
        "**Checked literature findings**", "", '| Mapped source              | Verified scope / correction                                                                                                                                                             |\n|:---------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|\n| Vocks et al 2021           | File attribution is incorrect: Knejzlikova et al. (2021), doi:10.3389/fpsyg.2021.673597. SCR used a log transform and SCL a square root; not evidence for blanket native-scale EDA.     |\n| Figner_Murphy_2010         | Local full text discusses log or square-root treatment of skewed SCR and within-participant standardization; a conditional method discussion, not proof that no transformation is best. |\n| EDA_recommendations_BIOPAC | Local 2012 recommendations discuss logarithmic or square-root SCR amplitudes; recording in microsiemens is not the same as analysing untransformed values.                              |\n| Laborde Mosley Thayer 2017 | Normality of HRV Parameters discusses natural-log RMSSD and absolute powers for non-normal HRV; this is general rationale, not variable-by-variable validation of our full model.       |\n\nThe local file labelled Vocks 2021 is actually [Knejzlikova et al. (2021)](https://doi.org/10.3389/fpsyg.2021.673597). Its main EDA analysis used log-transformed SCR and square-root SCL. It cannot be counted as support for blanket no-transformation EDA. Current H4 EDA rules are not silently changed by this documentation correction.', "",
        "| Exposure group | Current treatment | Literature and decision |", '|---|---|---|\n| PM1/PM2.5/PM10 | Native scale in current H4; ln(x) is a candidate sensitivity, not yet applied | [Biel et al. (2020)](https://doi.org/10.1038/s41598-020-73412-6) transformed UFP and black carbon selectively. That is not evidence for universally logging our PM size fractions. Compare physiological model form/residuals before changing equations. |\n| Lux / melanopic illuminance | Native scale currently; ln(x) candidate where positive | [Zauner et al. (2025)](https://doi.org/10.1177/07487304251336624) specifically addresses logarithmic light modelling and zeros. Their light-response examples do not mandate the same transform when lux predicts physiology. Do not add an arbitrary constant for zeros. |\n| Sound level | Calibrated dBA retained; no extra log | dBA already expresses a logarithmic level; standardizing is not another logarithm. |\n| Sound scores, proportions and Hz descriptors | Current correlation-PCA uses native values centered within bout and pooled-SD scaled | [Jolliffe & Cadima (2016)](https://doi.org/10.1098/rsta.2015.0202) supports standardized PCA; it does not prove native/log alternatives are equivalent. Overlapping representations remain a limitation. |\n| SVF / percentages / greenery | Native bounded predictors, then model-specific centering/scaling | Being bounded does not itself require a logit predictor. Endpoint and exposure-response checks are still needed; no universal SVF-transform claim is made. |\n| Temperature, indices, radiation and other physical predictors | Native units in current equations | Choose functional form for the scientific exposure-response relationship, not from a normality test on the predictor alone ([NIST](https://itl.nist.gov/div898/handbook/pmd/section4/pmd453.htm)). |\n| Diagnostic / category codes | Keep for review or use category coding; do not continuously standardize | Unidentified sound share is excluded before PCA. Ordinal codes are not made continuous by numeric storage. |', "",
        "Detailed CSVs retain the observation counts and distribution diagnostics; they are not repeated here.", ""])




def save_normalization_outputs(tables, report_text):
    payloads = {path: table.to_csv(index=False).encode("utf-8") for table, path in tables}
    payloads[OUT_RESULTS] = report_text.encode("utf-8")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(OUT_DIR).free < sum(map(len, payloads.values())) + 10_000_000:
        raise OSError("Insufficient space: existing outputs were not opened or replaced")
    staged = []
    try:
        for path, payload in payloads.items():
            with tempfile.NamedTemporaryFile(dir=OUT_DIR, suffix=".tmp", delete=False) as handle:
                staged.append((Path(handle.name), path))
                handle.write(payload)
                handle.flush()
                os.fsync(handle.fileno())
        for temporary, path in staged:
            os.replace(temporary, path)
            print(path.resolve())
    finally:
        for temporary, _ in staged:
            temporary.unlink(missing_ok=True)


In [5]:
# 4. Run
physio_counts, papers_table, physio = make_normalization_tables()
count = make_column_inventory(physio_counts)
paper_roles = make_paper_roles()
assert len(physio) == 27 and physio.Column.nunique() == 27
assert not physio["Transformation in Step 5"].str.contains(r"1 \+ x|log1p", regex=True).any()
assert set(source_columns) == set(count.loc[count.Origin == "Merged source", "col_name"])
print(f"Covered {len(source_columns)} source columns plus {len(AVERAGES)} bilateral means; {len(core)} core-phase rows.")
print("This notebook writes transformation decisions and literature tables, not transformed sensor data.")
display(count.groupby(["Role", "Transformation"], dropna=False).size().rename("Columns").reset_index())

display(count)
display(papers_table)
print("Empatica and eyetracker transformations")
display(physio)
environment = make_environment_table(count)
assert set(count.loc[count.Role == "Sound exposure", "col_name"]) == set(sound_pca_columns)
print(f"Environmental transformations: {len(environment)} fields, including all {len(sound_pca_columns)} sound PCA inputs")
display(environment)

Covered 224 source columns plus 6 bilateral means; 2156 core-phase rows.
This notebook writes transformation decisions and literature tables, not transformed sensor data.


,Role,Transformation,Columns
0,Categorical / ordinal code,No continuous transformation,7
1,Classification diagnostic,Not applicable to exposure model,1
2,Metadata,Not applicable,15
3,Numeric measurement / derived field,Native scale before model-specific centering/s...,76
4,Physiological outcome,Natural log,7
5,Physiological outcome,No transformation,20
6,QC / processing metadata,Not applicable,16
7,Questionnaire / participant metadata,Native score or categorical coding; no automat...,26
8,Questionnaire score,Native score or categorical coding; no automat...,12
9,SVF / greenery exposure,Native scale before model-specific centering/s...,17


,col_name,Parameter,Role,Transformation,Basis,Scaling,Number of papers,Papers,Origin,Core rows,Missing,Unique,Nonpositive,Minimum,Maximum,Skewness,Nonfinite
0,ParticipantID,ParticipantID,Metadata,Not applicable,"Identifier, timestamp or positioning metadata;...",Model-specific; metadata/QC/categorical fields...,0,,Merged source,2156,0,11,NaN,NaN,NaN,NaN,0
1,PhaseID,PhaseID,Metadata,Not applicable,"Identifier, timestamp or positioning metadata;...",Model-specific; metadata/QC/categorical fields...,0,,Merged source,2156,0,4,NaN,NaN,NaN,NaN,0
2,Datetime,Datetime,Metadata,Not applicable,"Identifier, timestamp or positioning metadata;...",Model-specific; metadata/QC/categorical fields...,0,,Merged source,2156,0,2156,NaN,NaN,NaN,NaN,0
3,Date,Date,Metadata,Not applicable,"Identifier, timestamp or positioning metadata;...",Model-specific; metadata/QC/categorical fields...,0,,Merged source,2156,0,11,NaN,NaN,NaN,NaN,0
4,empatica__met,empatica__met,Numeric measurement / derived field,Native scale before model-specific centering/s...,No additional nonlinear transform introduced. ...,Model-specific; metadata/QC/categorical fields...,0,,Merged source,2156,49,271,0.0,0.95,8.52,-0.310649,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,atmo_pm2.5_avg,atmo_pm2.5_avg,Numeric measurement / derived field,Native scale before model-specific centering/s...,No additional nonlinear transform introduced. ...,Model-specific; metadata/QC/categorical fields...,0,,Bilateral mean,2156,39,173,0.0,1.9,106.3,8.989033,0
226,atmo_pm10_avg,atmo_pm10_avg,Numeric measurement / derived field,Native scale before model-specific centering/s...,No additional nonlinear transform introduced. ...,Model-specific; metadata/QC/categorical fields...,0,,Bilateral mean,2156,39,291,0.0,2.4,971.0,12.325653,0
227,lys_lux_avg,lys_lux_avg,Numeric measurement / derived field,Native scale before model-specific centering/s...,No additional nonlinear transform introduced. ...,Model-specific; metadata/QC/categorical fields...,0,,Bilateral mean,2156,39,385,0.0,49.69614,37477.99115,2.346515,0
228,lys_medi_avg,lys_medi_avg,Numeric measurement / derived field,Native scale before model-specific centering/s...,No additional nonlinear transform introduced. ...,Model-specific; metadata/QC/categorical fields...,0,,Bilateral mean,2156,39,385,0.0,10.676689,6954.676279,1.112793,0


,Number,Paper title,File path,Evidence checked
0,[1],TEMP_EDA_HR - Zhang et al 2020 - Wearable Devi...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
1,[2],HR_EDA - Beggiato et al 2018 - Using Smartband...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
2,[3],Air temp - Outdoor temperature heart rate and ...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
3,[4],Personal_thermal_comfort_wearable_sensors_2018...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
4,[5],Thermal_comfort_physiological_sensing_DTU_2025...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
5,[6],LMM_physio_stress_Frontiers_2023_Tervonen.pdf,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
6,[7],Empatica - Kyriakou et al 2019 - An Individual...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
7,[8],Empatica - A novel methodology for human therm...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
8,[9],Psychophysiology - May 1981 - Fowles - Publica...,C:\Users\pandya\Documents\Github\docker\Paper3...,Existing mapping; not newly full-text verified
9,[10],EDA_recommendations_BIOPAC_Psychophysiology_20...,C:\Users\pandya\Documents\Github\docker\Paper3...,Local 2012 recommendations discuss logarithmic...


Empatica and eyetracker transformations


,Column,Parameter,Unit,Transformation in the papers,Transformation in Step 5,Rows excluded (x <= 0),Paper titles
0,empatica_heart_rate,Heart rate,bpm,Existing mapped references; native-scale optim...,No transformation,0,TEMP_EDA_HR - Zhang et al 2020 - Wearable Devi...
1,empatica_skin_temperature,Skin temperature,°C,Existing mapped references; native-scale optim...,No transformation,0,TEMP_EDA_HR - Zhang et al 2020 - Wearable Devi...
2,empatica__eda_scl_usiemens,Skin conductance level,microsiemens,Mixed / conditional: SCR log or square root; S...,Not in Step 5,0,Psychophysiology - May 1981 - Fowles - Publica...
3,empatica_eda_tonic,"Skin conductance, tonic",microsiemens,Mixed / conditional: SCR log or square root; S...,No transformation,0,Psychophysiology - May 1981 - Fowles - Publica...
4,empatica_eda_phasic,"Skin conductance, phasic",microsiemens,Mixed / conditional: SCR log or square root; S...,No transformation,0,Psychophysiology - May 1981 - Fowles - Publica...
5,empatica_hrv_td_sdnn,SDNN,ms,Natural-log HRV rationale; variable-specific m...,Natural log,0,HRV - Laborde Mosley Thayer 2017 - Heart Rate ...
6,empatica_hrv_td_rmssd,RMSSD,ms,Natural-log HRV rationale; variable-specific m...,Natural log,0,HRV - Laborde Mosley Thayer 2017 - Heart Rate ...
7,empatica_hrv_fd_lf,LF power,ms2,Natural-log HRV rationale; variable-specific m...,Natural log,0,HRV - Laborde Mosley Thayer 2017 - Heart Rate ...
8,empatica_hrv_fd_hf,HF power,ms2,Natural-log HRV rationale; variable-specific m...,Natural log,0,HRV - Laborde Mosley Thayer 2017 - Heart Rate ...
9,empatica_hrv_fd_vlf,VLF power,ms2,Natural-log HRV rationale; variable-specific m...,Natural log,0,HRV - Laborde Mosley Thayer 2017 - Heart Rate ...


Environmental transformations: 124 fields, including all 32 sound PCA inputs


,Column,Role,Nonlinear transformation,Centering / scaling,Condition / reason,Sound PCA input,Current Eq1 term,Specification,Finite observations,Nonpositive observations,Native skewness,Positive-only log skewness,Candidate alternative,Evidence scope
0,ucm_AIR_temp,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2156,0,0.066144,NaN,No additional nonlinear transform proposed aut...,"Native scale is the current specification, not..."
1,ucm_AIR_RH,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2156,0,0.122482,NaN,No additional nonlinear transform proposed aut...,"Native scale is the current specification, not..."
2,ucm_AIR_T_bot,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2156,0,0.083293,NaN,No additional nonlinear transform proposed aut...,"Native scale is the current specification, not..."
3,ucm_AIR_T_mid,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2156,0,0.180190,NaN,No additional nonlinear transform proposed aut...,"Native scale is the current specification, not..."
4,ucm_AIR_T_top,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2156,0,0.085610,NaN,No additional nonlinear transform proposed aut...,"Native scale is the current specification, not..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,atmo_pm2.5_avg,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,True,C:\Users\pandya\Documents\Github\docker\Paper3...,2117,0,8.989033,1.617084,ln(x) model-form sensitivity; not applied,"Biel et al. 2020, doi:10.1038/s41598-020-73412..."
120,atmo_pm10_avg,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2117,0,12.325653,0.739817,ln(x) model-form sensitivity; not applied,"Biel et al. 2020, doi:10.1038/s41598-020-73412..."
121,lys_lux_avg,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,True,C:\Users\pandya\Documents\Github\docker\Paper3...,2117,0,2.346515,-0.829908,ln(x) model-form sensitivity; not applied,"Zauner et al. 2025, doi:10.1177/07487304251336..."
122,lys_medi_avg,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling wh...,Current specification; model form and residual...,False,False,C:\Users\pandya\Documents\Github\docker\Paper3...,2117,0,1.112793,-2.458885,ln(x) model-form sensitivity; not applied,"Zauner et al. 2025, doi:10.1177/07487304251336..."


In [6]:
# Save: stage every output before replacing existing files.
physio_vote_summary, physio_vote_detail, env_vote_summary, env_vote_detail = make_transformation_vote_table()
vote_export = pd.concat([
    physio_vote_summary.assign(Domain="Physiology"),
    physio_vote_detail.assign(Domain="Physiology detail"),
    env_vote_summary.assign(Domain="Environment"),
    env_vote_detail.assign(Domain="Environment detail"),
], ignore_index=True)
tables = [(count, OUT_COUNTS), (papers_table, OUT_INDEX), (physio, OUT_PHYSIO),
          (paper_roles, OUT_ROLES), (environment, OUT_ENVIRONMENT), (vote_export, OUT_VOTES)]
save_normalization_outputs(tables, make_normalization_report(
    count, physio, environment, physio_vote_summary, physio_vote_detail, env_vote_summary, env_vote_detail))

# Final review tables
physiology_review = physio.drop(columns=["Rows excluded (x <= 0)"]).merge(
    physio_counts[["col_name", "Number of papers", "Papers"]].rename(columns={"col_name": "Column"}),
    on="Column", how="left", validate="one_to_one")
assert physiology_review["Number of papers"].notna().all()
print("Physiology: paper counts are mapped references, not verified transformation endorsements.")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(physiology_review)
    display(environment)


C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_counts.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_paper_index.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_physio.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_paper_roles.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_H1toH4_environment.csv
C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\statistical_analysis\Normalization & Parameter Screening\Normalization\Normalization_results.md
Physiology: pa

,Column,Parameter,Unit,Transformation in the papers,Transformation in Step 5,Paper titles,Number of papers,Papers
0,empatica_heart_rate,Heart rate,bpm,Existing mapped references; native-scale optimum not established,No transformation,TEMP_EDA_HR - Zhang et al 2020 - Wearable Device for Measuring Students Cognitive Load based on EDA Body Temperature and HR.pdf | HR_EDA - Beggiato et al 2018 - Using Smartbands Pupillometry and Body Motion to Detect Discomfort in Automated Driving.pdf | Air temp - Outdoor temperature heart rate and blood pressure in Chinese adults.pdf | Personal_thermal_comfort_wearable_sensors_2018.pdf | Thermal_comfort_physiological_sensing_DTU_2025.pdf | LMM_physio_stress_Frontiers_2023_Tervonen.pdf,6,"[1], [2], [3], [4], [5], [6]"
1,empatica_skin_temperature,Skin temperature,°C,Existing mapped references; native-scale optimum not established,No transformation,TEMP_EDA_HR - Zhang et al 2020 - Wearable Device for Measuring Students Cognitive Load based on EDA Body Temperature and HR.pdf | Personal_thermal_comfort_wearable_sensors_2018.pdf | Thermal_comfort_physiological_sensing_DTU_2025.pdf | Empatica - Kyriakou et al 2019 - An Individual-Oriented Algorithm for Stress Detection in Wearable Sensor Measurements.pdf | Empatica - A novel methodology for human thermal comfort decoding via physiological signals.pdf,5,"[1], [4], [5], [7], [8]"
2,empatica__eda_scl_usiemens,Skin conductance level,microsiemens,Mixed / conditional: SCR log or square root; SCL square root in the checked 2021 paper,Not in Step 5,Psychophysiology - May 1981 - Fowles - Publication Recommendations for Electrodermal Measurements.pdf | EDA_recommendations_BIOPAC_Psychophysiology_2012.pdf | EDA - Vocks et al 2021 - Electrodermal Response to Mirror Exposure in Restrictive Anorexia.pdf | TEMP_EDA_HR - Zhang et al 2020 - Wearable Device for Measuring Students Cognitive Load based on EDA Body Temperature and HR.pdf | EDA_transform_Figner_Murphy_2010_skin_conductance.pdf,5,"[9], [10], [11], [1], [12]"
3,empatica_eda_tonic,"Skin conductance, tonic",microsiemens,Mixed / conditional: SCR log or square root; SCL square root in the checked 2021 paper,No transformation,Psychophysiology - May 1981 - Fowles - Publication Recommendations for Electrodermal Measurements.pdf | EDA_recommendations_BIOPAC_Psychophysiology_2012.pdf | EDA - Vocks et al 2021 - Electrodermal Response to Mirror Exposure in Restrictive Anorexia.pdf | TEMP_EDA_HR - Zhang et al 2020 - Wearable Device for Measuring Students Cognitive Load based on EDA Body Temperature and HR.pdf | EDA_transform_Figner_Murphy_2010_skin_conductance.pdf,5,"[9], [10], [11], [1], [12]"
4,empatica_eda_phasic,"Skin conductance, phasic",microsiemens,Mixed / conditional: SCR log or square root; SCL square root in the checked 2021 paper,No transformation,Psychophysiology - May 1981 - Fowles - Publication Recommendations for Electrodermal Measurements.pdf | EDA_recommendations_BIOPAC_Psychophysiology_2012.pdf | EDA - Vocks et al 2021 - Electrodermal Response to Mirror Exposure in Restrictive Anorexia.pdf | EDA_transform_BIOPAC_EDA_SCR_Analysis.pdf | EDA_transform_Figner_Murphy_2010_skin_conductance.pdf,5,"[9], [10], [11], [13], [12]"
5,empatica_hrv_td_sdnn,SDNN,ms,Natural-log HRV rationale; variable-specific mapping not fully verified,Natural log,HRV - Laborde Mosley Thayer 2017 - Heart Rate Variability and Cardiac Vagal Tone in Psychophysiological Research Recommendations.pdf | Empatica - Duration - An Overview of Heart Rate Variability Metrics and Norms.pdf | Empatica - HRV_task_force_team_europe_1996-1.pdf | Empatica - Reliability of Ultra-Short-Term Analysis as a Surrogate of Standard 5-Min Analysis of Heart Rate Variability.pdf | Impact of Using Different Levels of Threshold-Based Artefact Correction on the Quantification of Heart Rate Variability in Three Independent Human Cohorts.pdf,5,"[14], [15], [16], [17], [18]"
6,empatica_hrv_td_rmssd,RMSSD,ms,Natural-log HRV rationale; variable-specific mapping not fully 

,Column,Role,Nonlinear transformation,Centering / scaling,Condition / reason,Sound PCA input,Current Eq1 term,Specification,Finite observations,Nonpositive observations,Native skewness,Positive-only log skewness,Candidate alternative,Evidence scope
0,ucm_AIR_temp,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.066144,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
1,ucm_AIR_RH,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.122482,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
2,ucm_AIR_T_bot,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.083293,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
3,ucm_AIR_T_mid,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.180190,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
4,ucm_AIR_T_top,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.085610,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
5,ucm_IR_back,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.739794,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
6,ucm_IR_front,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.234060,NaN,No additional nonlinear transform proposed automatically,"Native scale is the current specification, not an optimized or universally validated choice"
7,ucm_IR_left,Numeric measurement / derived field,None; native scale,Within-bout centering and pooled-SD scaling when entered in H4,Current specification; model form and residual assumptions require separate checks,False,False,C:\Users\pandya\Documents\Github\docker\Paper3_Final\scripts\H4_multivariate.R,2156,0,0.106134,NaN,No additional nonlinear transform proposed automatically,"Native scale is the c